In [3]:

import json
import time
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Set

import pandas as pd
import requests


class PncpSearchPremiumClient:
    """
    Cliente PNCP com fatiamento por UF usando o parâmetro correto `ufs=<sigla>`.
    Estado encapsulado na instância: cache, IDs vistos, pasta de saída.
    """

    BASE_URL = "https://pncp.gov.br/api/search/"
    BASE_SITE_URL = "https://pncp.gov.br"
    ESTADOS_BR: List[str] = [
        "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
        "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
        "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
    ]
    # teto seguro: se collect_range atingir isso, subdivide o intervalo
    MAX_ITEMS_PER_QUERY = 10_000

    # ------------------------------------------------------------------
    # Inicialização
    # ------------------------------------------------------------------
    def __init__(
        self,
        timeout: int = 60,
        sleep_seconds: float = 0.25,
        output_dir: Optional[Path] = None,
        session: Optional[requests.Session] = None,
        debug: bool = False,          # ativa logs de URL/params
    ) -> None:
        self.timeout = timeout
        self.sleep_seconds = sleep_seconds
        self.debug = debug

        # Estado interno — sem variáveis globais
        self._ids_vistos: Set[str] = set()
        self._query_cache: Dict[str, Dict[str, Any]] = {}
        self._processed_ranges: Set[str] = set()

        # Pasta de saída com timestamp único por instância
        ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        self.pasta_saida = output_dir or (Path("data") / ts)
        self.pasta_saida.mkdir(parents=True, exist_ok=True)
        self.arquivo_jsonl = self.pasta_saida / "pncp_resultado.jsonl"
        self.arquivo_csv   = self.pasta_saida / "pncp_resultado.csv"

        self.session = session or requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0",
            "Accept": "application/json",
        })

        print(f"📁 Saída incremental em: {self.pasta_saida}")

    # ------------------------------------------------------------------
    # HTTP — com log de debug de params e URL real
    # ------------------------------------------------------------------
    def _get(self, params: Dict[str, Any]) -> Dict[str, Any]:
        if self.debug:
            print(f"  [DEBUG params] {params}")

        try:
            response = self.session.get(
                self.BASE_URL, params=params, timeout=self.timeout
            )
        except requests.RequestException as exc:
            print(f"[ERRO] Falha na requisição: {exc}")
            return {"items": [], "total": 0, "_http_error": True}

        if self.debug:
            print(f"  [DEBUG URL]   {response.url}")

        if response.status_code == 400:
            return {"items": [], "total": 0, "_http_400": True}

        response.raise_for_status()
        return response.json()

    def _build_url(self, item_url: Optional[str]) -> Optional[str]:
        if not item_url:
            return None
        if item_url.startswith(("http://", "https://")):
            return item_url
        return f"{self.BASE_SITE_URL}{item_url}"

    # ------------------------------------------------------------------
    # Normalização
    # ------------------------------------------------------------------
    def normalizar_item(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "objetoCompra":             item.get("description"),
            "titulo":                   item.get("title"),
            "id":                       item.get("id"),
            "numeroControlePNCP":       item.get("numero_controle_pncp"),
            "anoCompra":                item.get("ano"),
            "sequencialCompra":         item.get("numero_sequencial"),
            "numero":                   item.get("numero"),
            "documentType":             item.get("document_type"),
            "tipoId":                   item.get("tipo_id"),
            "tipoNome":                 item.get("tipo_nome"),
            "modalidadeLicitacaoId":    item.get("modalidade_licitacao_id"),
            "modalidadeLicitacaoNome":  item.get("modalidade_licitacao_nome"),
            "situacaoId":               item.get("situacao_id"),
            "situacaoNome":             item.get("situacao_nome"),
            "orgaoId":                  item.get("orgao_id"),
            "orgaoCnpj":                item.get("orgao_cnpj"),
            "orgaoNome":                item.get("orgao_nome"),
            "unidadeId":                item.get("unidade_id"),
            "unidadeCodigo":            item.get("unidade_codigo"),
            "unidadeNome":              item.get("unidade_nome"),
            "esferaId":                 item.get("esfera_id"),
            "esferaNome":               item.get("esfera_nome"),
            "poderId":                  item.get("poder_id"),
            "poderNome":                item.get("poder_nome"),
            "municipioId":              item.get("municipio_id"),
            "municipioNome":            item.get("municipio_nome"),
            "uf":                       item.get("uf"),
            "dataPublicacaoPncp":       item.get("data_publicacao_pncp"),
            "dataAtualizacaoPncp":      item.get("data_atualizacao_pncp"),
            "dataInicioVigencia":       item.get("data_inicio_vigencia"),
            "dataFimVigencia":          item.get("data_fim_vigencia"),
            "dataAssinatura":           item.get("data_assinatura"),
            "cancelado":                item.get("cancelado"),
            "temResultado":             item.get("tem_resultado"),
            "valorGlobal":              item.get("valor_global"),
            "fonteOrcamentaria":        item.get("fonte_orcamentaria"),
            "fonteOrcamentariaId":      item.get("fonte_orcamentaria_id"),
            "fonteOrcamentariaNome":    item.get("fonte_orcamentaria_nome"),
            "itemUrl":                  item.get("item_url"),
            "urlPncp":                  self._build_url(item.get("item_url")),
            "createdAt":                item.get("createdAt"),
        }

    # ------------------------------------------------------------------
    # Persistência incremental (privados)
    # ------------------------------------------------------------------
    def _append_jsonl(self, items: List[Dict[str, Any]]) -> None:
        if not items:
            return
        with open(self.arquivo_jsonl, "a", encoding="utf-8") as f:
            for item in items:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    def _append_csv(self, items: List[Dict[str, Any]]) -> None:
        if not items:
            return
        df = pd.json_normalize(items)
        header = not self.arquivo_csv.exists()
        df.to_csv(
            self.arquivo_csv,
            mode="a",
            header=header,
            index=False,
            encoding="utf-8-sig",
        )

    def _salvar_incremental(self, items: List[Dict[str, Any]]) -> None:
        self._append_jsonl(items)
        self._append_csv(items)

    # ------------------------------------------------------------------
    # Paginação básica
    # ------------------------------------------------------------------
    def buscar_pagina(
        self,
        pagina: int = 1,
        tam_pagina: int = 100,
        tipos_documento: Optional[List[str]] = None,
        status: Optional[str] = None,
        ordenacao: str = "-data",
        params_extras: Optional[Dict[str, Any]] = None,
    ) -> Dict[str, Any]:
        params: Dict[str, Any] = {
            "pagina": pagina,
            "tam_pagina": tam_pagina,
            "ordenacao": ordenacao,
        }
        if status:
            params["status"] = status
        if tipos_documento:
            params["tipos_documento"] = (
                tipos_documento[0] if len(tipos_documento) == 1
                else ",".join(tipos_documento)
            )
        if params_extras:
            params.update(params_extras)

        data = self._get(params)

        if data.get("_http_400") or data.get("_http_error"):
            return {
                "pagina": pagina, "tam_pagina": tam_pagina,
                "total": None, "quantidade_itens": 0,
                "items": [], "items_normalizados": [],
                "http_400": True,
            }

        items = data.get("items", [])
        return {
            "pagina": pagina,
            "tam_pagina": tam_pagina,
            "total": data.get("total", 0),
            "quantidade_itens": len(items),
            "items": items,
            "items_normalizados": [self.normalizar_item(x) for x in items],
            "http_400": False,
        }

    def buscar_todas_paginas(
        self,
        tam_pagina: int = 100,
        tipos_documento: Optional[List[str]] = None,
        status: Optional[str] = None,
        ordenacao: str = "-data",
        params_extras: Optional[Dict[str, Any]] = None,
        max_paginas: Optional[int] = None,
        verbose: bool = False,
    ) -> Dict[str, Any]:
        pagina = 1
        total_esperado: Optional[int] = None
        itens_norm: List[Dict[str, Any]] = []
        ids_locais: Set[str] = set()

        while True:
            if max_paginas is not None and pagina > max_paginas:
                break

            res = self.buscar_pagina(
                pagina=pagina,
                tam_pagina=tam_pagina,
                tipos_documento=tipos_documento,
                status=status,
                ordenacao=ordenacao,
                params_extras=params_extras,
            )

            if res["http_400"]:
                if verbose:
                    print(f"[INFO] HTTP 400 na página {pagina}. Encerrando.")
                break

            items      = res["items"]
            items_norm = res["items_normalizados"]
            total      = res["total"]

            if total_esperado is None and total is not None:
                total_esperado = total

            if verbose:
                print(
                    f"[INFO] p.{pagina} | página: {len(items)} | "
                    f"acumulado: {len(itens_norm)} | total API: {total_esperado}"
                )

            if not items:
                break

            for bruto, norm in zip(items, items_norm):
                item_id = bruto.get("id")
                if item_id and item_id in ids_locais:
                    continue
                if item_id:
                    ids_locais.add(item_id)
                itens_norm.append(norm)

            if len(items) < tam_pagina:
                break

            pagina += 1
            if self.sleep_seconds > 0:
                time.sleep(self.sleep_seconds)

        return {
            "total_informado_api": total_esperado,
            "total_coletado": len(itens_norm),
            "items": itens_norm,
        }

    # ------------------------------------------------------------------
    # Busca por palavra-chave (retorna DataFrame)
    # ------------------------------------------------------------------
    def buscar_por_palavra_chave(
        self,
        termo: str,
        tam_pagina: int = 100,
        tipos_documento: Optional[List[str]] = None,
        status: Optional[str] = "recebendo_proposta",
        ordenacao: str = "-data",
        max_paginas: Optional[int] = None,
        somente_colunas_principais: bool = True,
        verbose: bool = True,
    ) -> pd.DataFrame:
        """Busca por termo livre e retorna DataFrame normalizado."""
        resultado = self.buscar_todas_paginas(
            tam_pagina=tam_pagina,
            tipos_documento=tipos_documento or ["edital"],
            status=status,
            ordenacao=ordenacao,
            params_extras={"q": termo},
            max_paginas=max_paginas,
            verbose=verbose,
        )
        items = resultado["items"]
        if not items:
            print(f"[INFO] Nenhum resultado para: '{termo}'")
            return pd.DataFrame()

        df = pd.DataFrame(items)
        if somente_colunas_principais:
            cols = [
                "numeroControlePNCP", "objetoCompra", "orgaoNome",
                "municipioNome", "uf", "modalidadeLicitacaoNome",
                "situacaoNome", "dataPublicacaoPncp", "dataFimVigencia",
                "valorGlobal", "urlPncp",
            ]
            df = df[[c for c in cols if c in df.columns]]
        return df

    # ------------------------------------------------------------------
    # Coleta recursiva por intervalo de datas
    # ------------------------------------------------------------------
    def collect_range(
        self,
        start: date,
        end: date,
        tam_pagina: int = 100,
        depth: int = 0,
        max_depth: int = 6,
        verbose: bool = True,
        uf: Optional[str] = None,     # <-- UF como argumento dedicado (não params_extras)
    ) -> None:
        """
        Coleta editais em [start, end] com filtro opcional de UF.
        Usa o parâmetro correto `ufs=<sigla>` na API.
        Se >= MAX_ITEMS_PER_QUERY, divide o intervalo recursivamente.
        """
        if start > end:
            return

        # Constrói params_extras com o parâmetro correto da API: `ufs`
        base_extras: Dict[str, Any] = {
            "dataInicial": start.strftime("%Y%m%d"),
            "dataFinal":   end.strftime("%Y%m%d"),
        }
        if uf:
            base_extras["ufs"] = uf   # ← parâmetro correto confirmado na API

        # Chave de cache/processamento inclui UF para evitar colisões entre estados
        cache_key = json.dumps({
            "start":      start.strftime("%Y%m%d"),
            "end":        end.strftime("%Y%m%d"),
            "tam_pagina": tam_pagina,
            "ufs":        uf or "",
        }, sort_keys=True)

        if cache_key in self._processed_ranges:
            return

        # Aproveita cache de query se já fez essa consulta antes
        if cache_key in self._query_cache:
            resultado = self._query_cache[cache_key]
        else:
            resultado = self.buscar_todas_paginas(
                tam_pagina=tam_pagina,
                tipos_documento=["edital"],
                status="recebendo_proposta",
                ordenacao="-data",
                params_extras=base_extras,
                verbose=False,
            )
            self._query_cache[cache_key] = resultado

        coletados = resultado.get("total_coletado", 0)
        total_api = resultado.get("total_informado_api")
        prefixo   = "  " * depth
        uf_label  = f"[UF:{uf}] " if uf else ""

        if verbose:
            print(f"{prefixo}{uf_label}[{start} → {end}] API:{total_api} | coletados:{coletados} | depth={depth}")

        # --- Dentro do limite: salva e encerra ---
        if coletados < self.MAX_ITEMS_PER_QUERY:
            novos: List[Dict[str, Any]] = []
            for item in resultado["items"]:
                item_id = item.get("id")
                if item_id and item_id in self._ids_vistos:
                    continue
                if item_id:
                    self._ids_vistos.add(item_id)
                novos.append(item)

            if novos:
                self._salvar_incremental(novos)
                if verbose:
                    print(f"{prefixo}  ✔ {uf_label}+{len(novos)} novos (acumulado: {len(self._ids_vistos)})")

            self._processed_ranges.add(cache_key)
            return

        # --- Atingiu teto: tenta subdividir ---
        if depth >= max_depth:
            print(f"[WARN] {uf_label}{start}→{end}: teto + max_depth={max_depth}. Salvando parcial.")
            novos = []
            for item in resultado["items"]:
                item_id = item.get("id")
                if item_id and item_id in self._ids_vistos:
                    continue
                if item_id:
                    self._ids_vistos.add(item_id)
                novos.append(item)
            if novos:
                self._salvar_incremental(novos)
            self._processed_ranges.add(cache_key)
            return

        delta = (end - start).days
        if delta >= 1:
            mid = start + timedelta(days=delta // 2)
            self.collect_range(start, mid,                    tam_pagina, depth+1, max_depth, verbose, uf)
            self.collect_range(mid + timedelta(days=1), end,  tam_pagina, depth+1, max_depth, verbose, uf)
            self._processed_ranges.add(cache_key)
        else:
            print(f"[WARN] {uf_label}Dia único {start} com overflow. Salvando parcial.")
            novos = []
            for item in resultado["items"]:
                item_id = item.get("id")
                if item_id and item_id in self._ids_vistos:
                    continue
                if item_id:
                    self._ids_vistos.add(item_id)
                novos.append(item)
            if novos:
                self._salvar_incremental(novos)
            self._processed_ranges.add(cache_key)

    # ------------------------------------------------------------------
    # Coleta por lista de UFs — único responsável por iterar os estados
    # ------------------------------------------------------------------
    def collect_by_ufs(
        self,
        ufs: Optional[List[str]] = None,
        start: Optional[date] = None,
        end: Optional[date] = None,
        tam_pagina: int = 100,
        max_depth: int = 6,
        verbose: bool = True,
    ) -> None:
        """
        Itera sobre cada UF e executa collect_range com `ufs=<sigla>`.
        Parâmetro correto da API confirmado: `ufs` (plural).
        """
        ufs   = ufs   or self.ESTADOS_BR
        end   = end   or date.today()
        start = start or date.today()

        total_ufs = len(ufs)
        for i, uf in enumerate(ufs, 1):
            print(f"\n{'─'*52}")
            print(f"  [{i}/{total_ufs}] UF: {uf} | {start} → {end}")
            print(f"{'─'*52}")
            self.collect_range(
                start=start,
                end=end,
                tam_pagina=tam_pagina,
                depth=0,
                max_depth=max_depth,
                verbose=verbose,
                uf=uf,                # passa UF de forma dedicada
            )

        print(f"\n{'='*52}")
        print(f"✅ Coleta finalizada | {len(self._ids_vistos)} itens únicos")
        print(f"   JSONL : {self.arquivo_jsonl}")
        print(f"   CSV   : {self.arquivo_csv}")
        print(f"{'='*52}")

    # ------------------------------------------------------------------
    # Atalho de execução completa
    # ------------------------------------------------------------------
    def run(
        self,
        start: Optional[date] = None,
        end: Optional[date] = None,
        ufs: Optional[List[str]] = None,
        tam_pagina: int = 100,
        max_depth: int = 6,
        verbose: bool = True,
    ) -> None:
        """Ponto de entrada único: coleta por UFs no intervalo de datas."""
        self.collect_by_ufs(
            ufs=ufs,
            start=start,
            end=end,
            tam_pagina=tam_pagina,
            max_depth=max_depth,
            verbose=verbose,
        )

    # ------------------------------------------------------------------
    # Utilitários de persistência explícita (salvar_json / salvar_csv)
    # ------------------------------------------------------------------
    def salvar_json(
        self,
        resultado: Dict[str, Any],
        caminho_arquivo: str,
        somente_items: bool = False,
    ) -> None:
        conteudo = resultado["items"] if somente_items else resultado
        with open(caminho_arquivo, "w", encoding="utf-8") as f:
            json.dump(conteudo, f, ensure_ascii=False, indent=2)

    def salvar_csv(
        self,
        resultado: Dict[str, Any],
        caminho_arquivo: str,
        somente_colunas_principais: bool = False,
    ) -> None:
        df = pd.json_normalize(resultado["items"])
        if somente_colunas_principais:
            cols = [
                "objetoCompra", "titulo", "numeroControlePNCP", "anoCompra",
                "sequencialCompra", "tipoNome", "modalidadeLicitacaoNome",
                "situacaoNome", "orgaoNome", "unidadeNome", "municipioNome",
                "uf", "dataPublicacaoPncp", "dataAtualizacaoPncp",
                "dataInicioVigencia", "dataFimVigencia", "urlPncp",
            ]
            df = df[[c for c in cols if c in df.columns]]
        df.to_csv(caminho_arquivo, index=False, encoding="utf-8-sig")


In [4]:

from datetime import date

# Instancia o cliente — cria pasta data/<timestamp>/ automaticamente
client = PncpSearchPremiumClient(sleep_seconds=0.25)

# Coleta todos os estados do Brasil de 21/03/2026 até hoje
client.run(
    start=date(2026, 3, 21),
    end=date.today(),
    ufs=PncpSearchPremiumClient.ESTADOS_BR,
    tam_pagina=100,
    max_depth=6,
    verbose=True,
)


📁 Saída incremental em: data/2026-03-21_14-52-32

────────────────────────────────────────────────────
  [1/27] UF: AC | 2026-03-21 → 2026-03-21
────────────────────────────────────────────────────
[UF:AC] [2026-03-21 → 2026-03-21] API:139 | coletados:139 | depth=0
  ✔ [UF:AC] +139 novos (acumulado: 139)

────────────────────────────────────────────────────
  [2/27] UF: AL | 2026-03-21 → 2026-03-21
────────────────────────────────────────────────────
[UF:AC] [2026-03-21 → 2026-03-21] API:139 | coletados:139 | depth=0
  ✔ [UF:AC] +139 novos (acumulado: 139)

────────────────────────────────────────────────────
  [2/27] UF: AL | 2026-03-21 → 2026-03-21
────────────────────────────────────────────────────
[UF:AL] [2026-03-21 → 2026-03-21] API:342 | coletados:342 | depth=0
  ✔ [UF:AL] +342 novos (acumulado: 481)

────────────────────────────────────────────────────
  [3/27] UF: AP | 2026-03-21 → 2026-03-21
────────────────────────────────────────────────────
[UF:AP] [2026-03-21 → 2026-03-2

In [ ]:

from datetime import date

# Teste rápido: apenas SP, hoje, debug=True para ver URL e params na saída
client_teste = PncpSearchPremiumClient(sleep_seconds=0.1, debug=True)

client_teste.collect_by_ufs(
    ufs=["SP", "MG"],
    start=date.today(),
    end=date.today(),
    tam_pagina=100,
    max_depth=2,
    verbose=True,
)
